In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
BASE_DIR = Path.cwd()
PROJECT_DIR = BASE_DIR
DATA_DIR = PROJECT_DIR / "data"
MODEL_DIR = PROJECT_DIR / "models"

## 生成 V2 數學函數數據

In [3]:
x = np.linspace(-2, 2, 10)         # x 座標
rng = np.random.default_rng(42)

In [4]:
def generate_function(label, x, range):     # V2 函數產生器
    param_1 = np.nan
    param_2 = np.nan
    param_3 = np.nan
    param_4 = np.nan

    if label == "linear":                   # 1. Linear: y = ax + b
        a = range.uniform(-2, 2)
        b = range.uniform(-3, 3)
        param_1 = a
        param_2 = b
        y_true = a*x + b

    elif label == "quadratic":              # 2. Quadratic: y = ax^2 + bx + c
        a = range.uniform(-2, 2)
        if abs(a) < 0.1:
            a = 0.1 if a >= 0 else -0.1
        b = range.uniform(-2, 2)
        c = range.uniform(-3, 3)
        param_1 = a
        param_2 = b
        param_3 = c
        y_true = a*x**2 + b*x + c
        
    elif label == "cubic":                  # 3. Cubic: y = ax^3 + bx^2 + cx + d
        a = range.uniform(-2, 2)
        if abs(a) < 0.1:
            a = 0.1 if a >= 0 else -0.1
        b = range.uniform(-2, 2)
        c = range.uniform(-2, 2)
        d = range.uniform(-3, 3)
        param_1 = a
        param_2 = b
        param_3 = c
        param_4 = d
        y_true = a*x**3 + b*x**2 + c*x + d

    elif label == "exponential":            # 4. Exponential: y = a * exp(bx) + c
        a = range.uniform(-2.0, 2.0)
        if abs(a) < 0.1:
            a = 0.1 if a >= 0 else -0.1
        b = range.uniform(-3.0, 3.0)
        if abs(b) < 0.3:
            b = 0.3 if b >= 0 else -0.3
        c = range.uniform(-3, 3)
        param_1 = a
        param_2 = b
        param_3 = c
        y_true = a * np.exp(b*x) + c

    elif label == "logarithmic":            # 5. Logarithm: y = a * log(|x+b|) + c
        a = range.uniform(-2, 2)
        if abs(a) < 0.1:
            a = 0.1 if a >= 0 else -0.1
        b = range.uniform(2.1, 8.0)
        c = range.uniform(-3, 3)
        param_1 = a
        param_2 = b
        param_3 = c
        y_true = a * np.log(np.abs(x+b)) + c

    elif label == "sine":                   # 6. Sine: y = a * sin(bx + c) + d
        a = range.uniform(-3.0, 3.0)
        if abs(a) < 0.3:
            a = 0.3 if a >= 0 else -0.3
        b = range.uniform(0.1, 3.0)
        c = range.uniform(-np.pi/4, np.pi/4)
        d = range.uniform(-3, 3)
        param_1 = a
        param_2 = b
        param_3 = c
        param_4 = d
        y_true = a * np.sin(b*x+c) + d

    elif label == "cosine":                 # 7. Cosine: y = a * cos(bx + c) + d
        a = range.uniform(-3.0, 3.0)
        if abs(a) < 0.3:
            a = 0.3 if a >= 0 else -0.3
        b = range.uniform(0.1, 3.0)
        c = range.uniform(-np.pi/4, np.pi/4)
        d = range.uniform(-3, 3)
        param_1 = a
        param_2 = b
        param_3 = c
        param_4 = d
        y_true = a * np.cos(b*x+c) + d

    elif label == "reciprocal":             # 8. Reciprocal: y = a / (x + b) + c
        a = range.uniform(-3, 3)
        if abs(a) < 0.1:
            a = 0.1 if a >= 0 else -0.1
        b = range.uniform(2.5, 10.0)
        c = range.uniform(-2, 2)
        param_1 = a
        param_2 = b
        param_3 = c
        y_true = a/(x+b) + c
        
    else:
        raise ValueError(f"Unknown: {label}")

    # noise_sigma
    y_std = np.std(y_true)
    noise_ratio = range.uniform(0.02, 0.10)
    noise_sigma = y_std * noise_ratio
    noise_sigma = max(noise_sigma, 0.001)

    # 加入 Gaussian Noise
    noise = range.normal(loc=0, scale=noise_sigma, size=len(x))
    y = y_true + noise

    params = [param_1, param_2, param_3, param_4]
    return y, params, noise_sigma

In [5]:
labels = ["linear", "quadratic", "cubic", "exponential", 
          "logarithmic", "sine", "cosine", "reciprocal"]
dataset_rows = []
metadata_rows = []
sample_id = 1

In [6]:
for label in labels:
    for _ in range(500):
        y, params, noise_sigma = generate_function(label, x, rng)
        dataset_row = {"id": sample_id,
                       "label": label}
        
        for i in range(10):
            dataset_row[f"y_{i:02d}"] = y[i]
        dataset_row["noise"] = noise_sigma
        dataset_rows.append(dataset_row)
        metadata_row = {"id": sample_id,
                        "label": label,
                        "param_1": params[0],
                        "param_2": params[1],
                        "param_3": params[2],
                        "param_4": params[3],
                        "noise_sigma": noise_sigma}
        metadata_rows.append(metadata_row)

        sample_id += 1

In [7]:
# 建立DataFrame
dataset_df = pd.DataFrame(dataset_rows)

In [8]:
dataset_df.to_csv(f'{DATA_DIR}/v2/function_dataset_v2.csv', index=False)

In [9]:
metadata_df = pd.DataFrame(metadata_rows)

In [10]:
metadata_df.to_csv(f'{DATA_DIR}/v2/function_metadata_v2.csv', index=False)

## 查看基本資訊(EDA)

In [11]:
df_dataset = pd.read_csv(f'{DATA_DIR}/v2/function_dataset_v2.csv')

In [12]:
print(df_dataset.shape)     # 數據外型

(4000, 13)


In [13]:
df_dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      4000 non-null   int64  
 1   label   4000 non-null   str    
 2   y_00    4000 non-null   float64
 3   y_01    4000 non-null   float64
 4   y_02    4000 non-null   float64
 5   y_03    4000 non-null   float64
 6   y_04    4000 non-null   float64
 7   y_05    4000 non-null   float64
 8   y_06    4000 non-null   float64
 9   y_07    4000 non-null   float64
 10  y_08    4000 non-null   float64
 11  y_09    4000 non-null   float64
 12  noise   4000 non-null   float64
dtypes: float64(11), int64(1), str(1)
memory usage: 406.4 KB


In [14]:
print(df_dataset.head())                # 前5筆資料

   id   label      y_00      y_01      y_02      y_03      y_04      y_05  \
0   1  linear -2.441687 -2.313400 -1.745867 -1.081418 -0.649480 -0.125297   
1   2  linear -2.898405 -2.408145 -1.718736 -1.203354 -0.638046 -0.095462   
2   3  linear -2.468578 -2.518337 -2.591504 -2.651249 -2.712069 -2.763373   
3   4  linear -1.823563 -1.392915 -1.181800 -0.812489 -0.519271 -0.209718   
4   5  linear -1.342078 -0.848862 -0.357639  0.193170  0.901426  1.147216   

       y_06      y_07      y_08      y_09     noise  
0  0.257987  0.959956  1.434383  1.833111  0.124065  
1  0.598109  1.085258  1.641828  2.220421  0.062923  
2 -2.819716 -2.885710 -2.948864 -3.007941  0.005496  
3  0.141197  0.394433  0.730522  0.986499  0.075105  
4  1.862640  2.075920  2.736921  3.302721  0.111954  


In [15]:
print(df_dataset.columns.tolist())      # 欄位

['id', 'label', 'y_00', 'y_01', 'y_02', 'y_03', 'y_04', 'y_05', 'y_06', 'y_07', 'y_08', 'y_09', 'noise']


In [16]:
print(df_dataset.dtypes)                # 資料型態

id         int64
label        str
y_00     float64
y_01     float64
y_02     float64
y_03     float64
y_04     float64
y_05     float64
y_06     float64
y_07     float64
y_08     float64
y_09     float64
noise    float64
dtype: object


In [17]:
print(df_dataset.isnull().sum())        # 缺失值

id       0
label    0
y_00     0
y_01     0
y_02     0
y_03     0
y_04     0
y_05     0
y_06     0
y_07     0
y_08     0
y_09     0
noise    0
dtype: int64


In [18]:
print(df_dataset['label'].value_counts())       # 查看各函數類型數量

label
linear         500
quadratic      500
cubic          500
exponential    500
logarithmic    500
sine           500
cosine         500
reciprocal     500
Name: count, dtype: int64


In [19]:
print(df_dataset['label'].value_counts(normalize=True))   # 查看各函數類型比例

label
linear         0.125
quadratic      0.125
cubic          0.125
exponential    0.125
logarithmic    0.125
sine           0.125
cosine         0.125
reciprocal     0.125
Name: proportion, dtype: float64


In [20]:
df_metadata = pd.read_csv(f'{DATA_DIR}/v2/function_metadata_v2.csv')

In [21]:
df_metadata.info()

<class 'pandas.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           4000 non-null   int64  
 1   label        4000 non-null   str    
 2   param_1      4000 non-null   float64
 3   param_2      4000 non-null   float64
 4   param_3      3500 non-null   float64
 5   param_4      1500 non-null   float64
 6   noise_sigma  4000 non-null   float64
dtypes: float64(5), int64(1), str(1)
memory usage: 218.9 KB


In [22]:
print(df_metadata.head())               # 前5筆資料

   id   label   param_1   param_2  param_3  param_4  noise_sigma
0   1  linear  1.095824 -0.366729      NaN      NaN     0.124065
1   2  linear  1.291046 -0.339515      NaN      NaN     0.062923
2   3  linear -0.133116 -2.737177      NaN      NaN     0.005496
3   4  linear  0.679256 -0.377088      NaN      NaN     0.075105
4   5  linear  1.147698  0.989105      NaN      NaN     0.111954


In [23]:
print(df_metadata.columns.tolist())     # 欄位

['id', 'label', 'param_1', 'param_2', 'param_3', 'param_4', 'noise_sigma']


In [24]:
print(df_metadata.dtypes)               # 資料型態

id               int64
label              str
param_1        float64
param_2        float64
param_3        float64
param_4        float64
noise_sigma    float64
dtype: object


In [25]:
print(df_metadata.isnull().sum())       # 缺失值

id                0
label             0
param_1           0
param_2           0
param_3         500
param_4        2500
noise_sigma       0
dtype: int64


In [26]:
print(df_metadata['noise_sigma'].describe())    # 查看noise分布

count    4000.000000
mean        0.218595
std         0.820608
min         0.001000
25%         0.015749
50%         0.054827
75%         0.141768
max        17.386222
Name: noise_sigma, dtype: float64


## 清理資料

In [27]:
df1 = df_dataset.drop(columns=['id'])

In [28]:
df1 = df1.drop(columns=['noise'])

In [29]:
df1.to_csv(f'{DATA_DIR}/v2/function_dataset_cleaned_v2.csv', index=False)

In [30]:
print(df1.shape)        # 清理後數據的形狀

(4000, 11)


In [31]:
# 進行數據清理, 將遺失值用"0"填補
df_metadata.fillna(0, inplace=True)

,id,label,param_1,param_2,param_3,param_4,noise_sigma
0,1,linear,1.095824,-0.366729,0.000000,0.0,0.124065
1,2,linear,1.291046,-0.339515,0.000000,0.0,0.062923
2,3,linear,-0.133116,-2.737177,0.000000,0.0,0.005496
3,4,linear,0.679256,-0.377088,0.000000,0.0,0.075105
4,5,linear,1.147698,0.989105,0.000000,0.0,0.111954
...,...,...,...,...,...,...,...
3995,3996,reciprocal,-2.429534,7.399423,-1.302406,0.0,0.005093
3996,3997,reciprocal,-2.788916,3.135173,-0.289944,0.0,0.055901
3997,3998,reciprocal,-0.864742,8.303313,1.389866,0.0,0.001000
3998,3999,reciprocal,-1.662651,2.592163,0.474439,0.0,0.064086


In [32]:
# 以清理過後的資料來蓋掉原本metadata資料
df2 = df_metadata
df2.to_csv(f'{DATA_DIR}/v2/function_metadata_v2.csv', index=False)

In [33]:
print(df2.shape)        # 清理後metadata資料形狀

(4000, 7)
